### Measuring the purity:

1) In a classification setting, if we are classifying images of cats and dogs for example. If all images are cats, then our split has max. purity

2) If all images are dogs, split is also completely pure.



### Entropy -> Measure of impurity of the data

When split is 50/50 , Entropy is 1 = max entropy max impurity

In [1]:
from sklearn.datasets import make_classification

### In classification decision tree, we predict that each observiation belongs to the _most occuring_ class of the training observations in the region to which it belongs

- we are interested in not only in the class prediction corresponding to a particular terminal node region

- but also in the __class proportions__ among the training observation that fall into that region 

Classification error rate is simply the fraction of training observations in that region that do not belong to the most common class:

E = 1 - max(Pmk) - > This error rate is not sufficient for the three growing so we take in consideration Entropy index

Entropy = - SUM (Pmk)logPmk

Entropy value will be close to zero when if the Pmk are all near zero or near one

### Choosing over the best split:

- which split reduces the entropy the most!

- reduction of entropy = INFORMATION GAIN

### Information gain is metric that shows us the reduction in Entropy

### It is important to mention that in algorithm, many splits will be made but algorithm decides which split to use by taking the split that has the **highest** information gain

### Information_gain = H(parent) - (weighted_right * H(p1) + weighted_left * H(p2))

- where **weighted_** average entropy in that node, because number of observations matter

- It is much worse to have high entropy with many observations than high entropy in lower number of observations

## **Choosing the best split**

- I will make a small custom dataset, and try to implement by hand, how DecisionTree work _under the hood_ for choosing the best split!

In [2]:
import pandas as pd
import numpy as np

In [ ]:
# last column is y
# 3 features to classify cat vs dog
# y = 1 -> cat
# y = 0 -> dog

dataset_custom = pd.DataFrame([
    [1, 1, 1,1],
    [0, 0, 1,1],
    [0, 1, 0,0],
    [1, 0, 1,0],
    [1, 1, 1,1],
    [1, 1, 0,1],
    [0, 0, 0,0],
    [1, 1, 0,1],
    [0, 1, 0,0],
    [0, 1, 0,0]]).rename(columns={0:'ear shape',1:'face shape',2:'whiskers',3:'target'})

dataset_custom

,ear shape,face shape,whiskers,target
0,1,1,1,1
1,0,0,1,1
2,0,1,0,0
3,1,0,1,0
4,1,1,1,1
5,1,1,0,1
6,0,0,0,0
7,1,1,0,1
8,0,1,0,0
9,0,1,0,0


In [54]:
features = dataset_custom.columns[:-1]
target = dataset_custom.columns[-1]

In [56]:
X_train = dataset_custom.loc[:,features]
y_train = dataset_custom.loc[:,target]

In [59]:
X_train

,ear shape,face shape,whiskers
0,1,1,1
1,0,0,1
2,0,1,0
3,1,0,1
4,1,1,1
5,1,1,0
6,0,0,0
7,1,1,0
8,0,1,0
9,0,1,0


### Function that calculates the entropy:

- Simply, if p is pure split such as 0 or 1, the entropy will be 0

- If p is 0.5 , entropy will be MAX (1)

In [68]:
total_class = len(y_train)

def get_entropy(p):
    
    if p == 1 or p == 0:
        return 0
    else:
        return - p*np.log2(p) - (1-p)*np.log2(1-p)
        
z = get_entropy(0.5)
print(z)

1.0


In [103]:
print(y_train.value_counts()) #Parent node - Maximum Impurity => 1
dataset_custom

target
1    5
0    5
Name: count, dtype: int64


,ear shape,face shape,whiskers,target
0,1,1,1,1
1,0,0,1,1
2,0,1,0,0
3,1,0,1,0
4,1,1,1,1
5,1,1,0,1
6,0,0,0,0
7,1,1,0,1
8,0,1,0,0
9,0,1,0,0


### Function that splits node by features:

In [144]:
X_train['ear shape']

0    1
1    0
2    0
3    1
4    1
5    1
6    0
7    1
8    0
9    0
Name: ear shape, dtype: int64

In [180]:
dataset_custom.index[X_train['ear shape']==1]

Index([0, 3, 4, 5, 7], dtype='int64')

In [211]:
def split_nodes(X_train,feature_name):
    
    left_indices = []
    right_indices = []
    
    if feature_name in X_train.columns:
        left_indices = X_train[feature_name] == 1
        right_indices = X_train[feature_name] == 0
        return dataset_custom.index[left_indices],dataset_custom.index[right_indices]
    else:
        return 'No selected column'
    
z = split_nodes(X_train,'ear shape')
print(z)

(Index([0, 3, 4, 5, 7], dtype='int64'), Index([1, 2, 6, 8, 9], dtype='int64'))


In [214]:
y_train[z[0]]

0    1
3    0
4    1
5    1
7    1
Name: target, dtype: int64

In [219]:
def weighted_entropy(X_train,y_train,feature_name):
    
    # sample/total sample * H(p1) - sample/total sample * H(1-p1)

    left,right = split_nodes(X_train,feature_name)
    
    w_left = len(left)/len(X_train)
    w_right = len(right)/len(X_train)

    p_left = sum(y_train[left]/len(left))
    p_right = sum(y_train[right]/len(right))

    weighted_entropy = w_left * get_entropy(p_left) + w_right* get_entropy(p_right)

    return weighted_entropy

x = weighted_entropy(X_train,y_train,'ear shape')
print(x)


0.7219280948873623


### Collect all weighted entropies:


In [229]:
w_entropy_list = []
for f in X_train.columns:
    res = weighted_entropy(X_train,y_train,f)
    w_entropy_list.append(res.round(2))
print(w_entropy_list)

[np.float64(0.72), np.float64(0.97), np.float64(0.88)]


## **Information gain**

- splitting on the parent node

In [232]:
y_train.value_counts()

target
1    5
0    5
Name: count, dtype: int64